Group: Modeling
===============

This script models an example strong lens on the 'group' scale, which typically have one or more "main" lens galaxies
and smaller extra galaxies nearby, whose light may blur with the source light and whose mass contributes significantly
to the ray-tracing, meaning both are therefore included in the strong lens model.

This example uses a list-based model composition API, where:

 - Main lens galaxies are built in a loop over centres loaded from a JSON file and stored in the model as
   `lens_0`, `lens_1`, etc. The group's one `ExternalShear` is held in an `al.MassField` in the model's `fields`
   slot, beside `galaxies`.

 - Extra galaxies are built in a loop over centres loaded from a separate JSON file and stored in an
   `extra_galaxies` collection. Their mass centres are fixed to the observed centres of light and their
   Einstein radii are given a uniform prior.

This list-based approach scales naturally to systems with many main lens galaxies and many extra galaxies.
The centres are loaded from JSON files (`main_lens_centres.json` and `extra_galaxies_centres.json`) rather
than being hardcoded, so the same script works for different datasets without code changes.

__Contents__

- **Scaling Relations:** This example models the mass of each galaxy individually, which means the number of dimensions of.
- **Group Halo:** The model contains no group-scale dark-matter halo — an explicit modelling choice.
- **Example:** This script fits an `Imaging` dataset of a 'group-scale' strong lens where.
- **Simulation:** Overview of how the simulated dataset was generated.
- **Data Preparation:** Data standards required for fitting with PyAutoLens.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Main Galaxies and Extra Galaxies:** For a group-scale lens, we designate there to be two types of lens galaxies in the system.
- **Centres:** The centres of both the main lens galaxies and the extra galaxies are loaded from JSON files in the.
- **Redshifts:** In this example all line of sight galaxies are at the same redshift as the lens galaxy, meaning.
- **Model:** Compose the lens model fitted to the data.
- **Coordinates:** Coordinate system assumptions for the model-fit.
- **Improved Lens Model:** The previous model used Sersic light profiles for the lens, source and extra galaxies.
- **Linear Light Profiles:** The MGE model below uses a **linear light profile** for the bulge and disk via the ``lp_linear``.
- **Over Sampling:** Set up the adaptive over-sampling grid for accurate light profile evaluation.
- **Search:** Configure the non-linear search used to fit the model.
- **Unique Identifier:** In the path above, the `unique_identifier` appears as a collection of characters, where this.
- **Iterations Per Update:** Every `iterations_per_quick_update`, the non-linear search outputs the maximum likelihood model and.
- **Live Visual Update:** Push the quick-update image to a live display surface.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **JAX:** JAX acceleration for fast GPU/CPU model-fitting.
- **VRAM Use:** When running AutoLens with JAX on a GPU, the analysis must fit within the GPU's available VRAM.
- **Run Times:** Profiling the expected run time of the model-fit.
- **Output Folder Layout:** Description of the structure of the `output` folder where results are written.
- **Result:** Overview of the results of the model-fit.
- **Features:** The examples in the `autolens_workspace/*/imaging/features` package illustrate other lens modeling.
- **HowToLens:** This `start_here.py` script, and the features examples above, do not explain many details of how.
- **Modeling Customization:** The folders `autolens_workspace/*/guides/modeling/searches` gives an overview of alternative.

__Scaling Relations__

This example models the mass of each galaxy individually, which means the number of dimensions of the model increases
as we model group scale lenses with more galaxies. This can lead to a model that is slow to fit and poorly constrained.
There may also not be enough information in the data to constrain every galaxy's mass.

A common approach to overcome this is to put many of the extra galaxies a scaling relation, where the mass of the
galaxies are related to their light via a observationally motivated scaling relation. This means that as more
galaxies are included in the lens model, the dimensionality of the model does not increase. Furthermore, their
luminosities act as priors on their masses, which helps ensure the model is well constrained.

Lens modeling using scaling relations is fully support and described in the `features/scaling_relation/modeling.ipynb` example.
If your group has many extra galaxies (e.g. more than 5) you probably want to read this example once you are confident
with this one.

__Group Halo__

The model in this example contains **no group-scale dark-matter halo**: all of the mass is in the galaxies. For
the simulated dataset fitted here that is the true model. For a real group, whether to add a shared halo is an
**explicit modelling choice** — some groups are well described by their galaxies alone, others (with larger
Einstein radii or image configurations no galaxy-mass model reproduces) need a dominant halo component.

The `features/group_halo` example teaches that decision: it fits the same data with and without a halo and
compares them via the lensing-mass radius, Bayesian evidence and external information. Whatever that choice,
the extra galaxies themselves are modeled as tidally truncated `dPIEMassSph` profiles (truncation encodes
stripping of a member's outer dark matter by the group environment's tidal field) — see the `__Model__` section
below. A published real-data example of group modeling with **PyAutoLens** is CSWA 19 (Ding et al. 2025), whose
model combines a group halo with scaling-relation members.

__Example__

This script fits an `Imaging` dataset of a 'group-scale' strong lens where

 - There is a main lens galaxy whose lens galaxy's light is an MGE.
 - There is a main lens galaxy whose total mass distribution is an `Isothermal`; the group's external shear is
   an `ExternalShear` held in an `al.MassField`.
 - There are two extra lens galaxies whose light models are `SersicSph` profiles and total mass distributions
   are tidally truncated `dPIEMassSph` models.
 - The source galaxy's light is an MGE.

__Simulation__

This script fits a simulated `Imaging` dataset of a strong lens, which is produced in the
script `autolens_workspace/*/imaging/simulator.py`

__Data Preparation__

The `Imaging` dataset fitted in this example confirms to a number of standard that make it suitable to be fitted in
**PyAutoLens**.

If you are intending to fit your own strong lens data, you will need to ensure it conforms to these standards, which are
described in the script `autolens_workspace/*/imaging/data_preparation/start_here.ipynb`.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Dataset__

Load the strong lens group dataset `simple`, which is the dataset we will use to perform lens modeling.

This is loaded via .fits files, which is a data format used by astronomers to store images.

The `pixel_scales` define the arc-second to pixel conversion factor of the image, which for the dataset we are using
is 0.1" / pixel.

In [ ]:
dataset_name = "simple"
dataset_path = Path("dataset", "group", dataset_name)

__Dataset Auto-Simulation__

If the dataset does not already exist on your system, it will be created by running the corresponding
simulator script. This ensures that all example scripts can be run without manually simulating data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/group/simulator.py"],
        check=True,
    )

dataset = al.Imaging.from_fits(
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    pixel_scales=0.1,
)

Use an `aplt.subplot_imaging_dataset` the plot the data, including:

 - `data`: The image of the strong lens.
 - `noise_map`: The noise-map of the image, which quantifies the noise in every pixel as their RMS values.
 - `psf`: The point spread function of the image, which describes the blurring of the image by the telescope optics.
 - `signal_to_noise_map`: Quantifies the signal-to-noise in every pixel.

In [ ]:
aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

The model-fit requires a 2D mask defining the regions of the image we fit the lens model to the data.

We create a 7.5 arcsecond circular mask and apply it to the `Imaging` object that the lens model fits.

In [ ]:
mask_radius = 7.5

mask = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    radius=mask_radius,
)

dataset = dataset.apply_mask(mask=mask)

If we plot the masked data, the mask removes the exterior regions of the image where there is no emission from the
lens and lensed source galaxies.

The mask used to fit the data can be customized, as described in
the script `autolens_workspace/*/guides/modeling/customize`

In [ ]:
aplt.subplot_imaging_dataset(dataset=dataset)

__Main Galaxies and Extra Galaxies__

For a group-scale lens, we designate there to be two types of lens galaxies in the system:

 - `main_galaxies`: The main lens galaxies which likely make up the majority of light and mass in the lens system.
 These are modeled individually and stored as `lens_0`, `lens_1`, etc. in the model's `galaxies` collection.
 Their centres are loaded from the `main_lens_centres.json` file. The group's one `ExternalShear` is held in an
 `al.MassField` in the model's `fields` slot.

 - `extra_galaxies`: The extra galaxies which are nearby the lens system and contribute to the lensing of the source
  galaxy. These are modeled with a more restrictive model, for example with their centres fixed to the observed
  centre of light and their mass distributions modeled using a scaling relation. These are grouped into a single
  `extra_galaxies` collection. Their centres are loaded from the `extra_galaxies_centres.json` file.

In this simple example group scale lens, there is one main lens galaxy and two extra galaxies.

__Centres__

The centres of both the main lens galaxies and the extra galaxies are loaded from JSON files in the dataset
directory. This makes the script reusable across different datasets without hardcoding centre values.

For the main lens galaxies, the centres are loaded from `main_lens_centres.json` (e.g. `[(0.0, 0.0)]`).
For the extra galaxies, the centres are loaded from `extra_galaxies_centres.json` (e.g. `[(3.5, 2.5), (-4.4, -5.0)]`).

If the centres of the extra galaxies are treated as free parameters, there are too many
parameters and the model may not be fitted accurately.

For group-scale lenses we therefore manually specify the centres of the extra galaxies, which are fixed to the observed
centres of light of the galaxies.

In a real analysis, one must determine the centres of the galaxies before modeling them, which can be done as follows:

 - Use the GUI tool in the `imaging/data_preparation/gui/extra_galaxies_centres.py` script to determine the centres
   of the extra galaxies.

 - Use image processing software like Source Extractor (https://sextractor.readthedocs.io/en/latest/).

 - Fit every galaxy individually with a light profile (e.g. an `Sersic`).

__Redshifts__

In this example all line of sight galaxies are at the same redshift as the lens galaxy, meaning multi-plane lensing
is not used.

If you have redshift information on the line of sight galaxies and some of their redshifts are different to the lens
galaxy, you can easily extend this example below to perform multi-plane lensing.

You would simply define a `redshift_list` and use this to set up the extra `Galaxy` redshifts.

__Model__

We compose a lens model where:

  - The main lens galaxy's light is a `Sersic` light profile [7 parameters].

 - The main lens galaxy's total mass distribution is an `Isothermal`; the group's external shear is an
   `ExternalShear` held in a `MassField` [7 parameters].

 - There are two extra lens galaxies with linear `SersicSph` light and tidally truncated `dPIEMassSph` total mass
   distributions, with centres fixed to the observed centres of light and one free `sigma` each (`r_core` and
   `r_cut` fixed) [8 parameters].

 - The source galaxy's light is a point `SersicCore` [6 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=28.

__Model Composition (List-Based API)__

The API below for composing a lens model uses the list-based approach, where main lens galaxies are built in a loop
and stored as `lens_0`, `lens_1`, etc. Extra galaxies are similarly built in a loop and collected into an
`extra_galaxies` collection.

This list-based API scales naturally: adding more main lens galaxies or extra galaxies simply means adding more
entries to the respective JSON files. The model composition code does not need to change.

The extra galaxies use the tidally truncated `dPIEMassSph` mass profile, not the untruncated `IsothermalSph`
used at galaxy scale (see `imaging/features/extra_galaxies`). This is the standard convention of group- and
cluster-scale modeling: member galaxies orbiting in the shared group potential have the outer parts of their
dark matter halos stripped by tides, so their profiles are physically truncated at a radius `r_cut`. The dPIE
is parameterized in Lenstool's native convention — `sigma` (fiducial velocity dispersion, km/s), `r_core` and
`r_cut` (arcsec) — with `sigma` each galaxy's one free parameter; `r_core` is fixed to zero (where the dPIE
is analytic) and `r_cut` to a fiducial value, and the redshifts and cosmology constants are pinned so they do
not inherit default priors and float.

A full description of model composition is provided by the model cookbook:

https://pyautolens.readthedocs.io/en/latest/general/model_cookbook.html

__Coordinates__

The model fitting default settings assume that the lens galaxy centre is near the coordinates (0.0", 0.0").

If for your dataset the lens is not centred at (0.0", 0.0"), we recommend that you either:

 - Reduce your data so that the centre is (`autolens_workspace/*/data_preparation`).
 - Manually override the lens model priors (`autolens_workspace/*/guides/modeling/customize`).

In [ ]:
# Load centres from JSON files:

main_lens_centres = al.from_json(file_path=dataset_path / "main_lens_centres.json")
extra_galaxies_centres = al.from_json(
    file_path=dataset_path / "extra_galaxies_centres.json"
)

# Main Lens Galaxies:

lens_dict = {}

for i, centre in enumerate(main_lens_centres):

    bulge = af.Model(al.lp.Sersic)

    mass = af.Model(al.mp.Isothermal)

    lens_dict[f"lens_{i}"] = af.Model(
        al.Galaxy,
        redshift=0.5,
        bulge=bulge,
        mass=mass,
    )

# External Shear (an `al.MassField`, in the model's `fields` slot below):

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

# Extra Galaxies:

extra_galaxies_list = []

for centre in extra_galaxies_centres:

    # Extra Galaxy Light

    bulge = af.Model(al.lp.SersicSph)

    # Extra Galaxy Mass

    mass = af.Model(al.mp.dPIEMassSph)

    mass.centre = centre
    mass.sigma = af.UniformPrior(lower_limit=0.0, upper_limit=300.0)
    mass.r_core = 0.0  # vanishing core — fixed; the dPIE is analytic at r_core = 0
    mass.r_cut = 10.0  # truncation fixed at a fiducial radius
    mass.redshift_object = 0.5
    mass.redshift_source = 1.0
    mass.H0 = 67.66  # pinned: model constants, not parameters to sample
    mass.Om0 = 0.30966

    # Extra Galaxy

    extra_galaxy = af.Model(al.Galaxy, redshift=0.5, bulge=bulge, mass=mass)

    extra_galaxies_list.append(extra_galaxy)

extra_galaxies = af.Collection(extra_galaxies_list)

# Source:

bulge = af.Model(al.lp.SersicCore)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Overall Lens Model:

model = af.Collection(
    galaxies=af.Collection(**lens_dict, source=source),
    fields=field,
    extra_galaxies=extra_galaxies,
)

The `info` attribute shows the model in a readable format.

This shows the group scale model, with separate entries for each main lens galaxy (e.g. `lens_0`), the source galaxy
and the extra galaxies collection.

The `info` below may not display optimally on your computer screen, for example the whitespace between parameter
names on the left and parameter priors on the right may lead them to appear across multiple lines. This is a
common issue in Jupyter notebooks.

The`info_whitespace_length` parameter in the file `config/general.yaml` in the [output] section can be changed to
increase or decrease the amount of whitespace (The Jupyter notebook kernel will need to be reset for this change to
appear in a notebook).

In [ ]:
print(model.info)

The same model can also be visualized as a figure, making its structure easier to understand at a glance.

The figure shows how the model is organized: which parameters belong to each component, and whether they are free,
fixed, shared, linked by an expression, solved during the fit, or not configured. `model.info` provides the
corresponding numerical details, including the prior assigned to each free parameter and the value of each fixed
parameter.

In [ ]:
af.ModelPlotter(model).figure()

__Improved Lens Model__

The previous model used Sersic light profiles for the lens, source and extra galaxies. This makes the model API concise,
readable, and easy to follow.

However, single Sersic profiles perform poorly for most strong lenses. Symmetric profiles (e.g. elliptical Sersics)
typically leave significant residuals because they cannot capture the irregular and asymmetric morphology of real
galaxies (e.g. isophotal twists, radially varying ellipticity).

For the extra galaxies, each Sersic also introduces 5 non-linear parameters, which means that as we add more extra
galaxies the model becomes increasingly complex, making it difficult to fit accurately and efficiently.

This example therefore uses a lens model that combines two features, described in detail elsewhere (but a brief
overview is provided below):

- **Linear light profiles**  (see ``autolens_workspace/*/imaging/features/linear_light_profiles``)
- **Multi-Gaussian Expansion (MGE) light profiles**  (see ``autolens_workspace/*/imaging/features/multi_gaussian_expansion``)

These features avoid wasted effort trying to fit Sersic profiles to complex data, which is likely to fail unless the
lens is extremely simple. This does mean the model composition is more complex and as a user its a steeper learning
curve to understand the API, but its worth it for the improved accuracy and speed of lens modeling.

__Multi-Gaussian Expansion (MGE)__

A Multi-Gaussian Expansion (MGE) decomposes the lens and source light into ~50-100 Gaussians with varying ellipticities
and sizes. An MGE captures irregular features far more effectively than Sersic profiles, leading to more accurate lens m
odels.

Remarkably, modeling with MGEs is also significantly faster than using Sersics: they remain efficient in JAX (on CPU
or GPU), require fewer non-linear parameters despite their flexibility, and yield simpler parameter spaces that
sample in far fewer iterations.

The MGE is extremely important for group-scale lenses. Every time we add an extra galaxy, the MGE does not add
any extra non-linear parameters, unlike light profiles like Sersics. This means we can model the light of many
extra galaxies, ensuring the lens light model is accurate, without making the model slow to fit or poorly constrained.

__Linear Light Profiles__

The MGE model below uses a **linear light profile** for the bulge and disk via the ``lp_linear`` API, instead of the
standard ``lp`` light profiles used above.

A linear light profile solves for the *intensity* of each component via a linear inversion, rather than treating it as
a free parameter. This reduces the dimensionality of the non-linear parameter space: a model with ~80 Gaussians
does not introduce ~80 additional free parameters.

Linear light profiles therefore improve speed and accuracy, and they are used by default in all modeling example.

__List-Based MGE Model__

The improved model below uses the same list-based API as the simple model above, but replaces the Sersic light
profiles with MGE models created via `al.model_util.mge_model_from`. The main lens galaxies use 20 Gaussians
with uniform centre priors, while extra galaxies use 10 Gaussians with centres fixed to the observed positions.

In [ ]:
# Main Lens Galaxies:

lens_dict = {}

for i, centre in enumerate(main_lens_centres):

    bulge = al.model_util.mge_model_from(
        mask_radius=mask_radius,
        total_gaussians=20,
        centre_prior_is_uniform=True,
        sigma_min=dataset.pixel_scales[0] / 10.0,
    )

    mass = af.Model(al.mp.Isothermal)

    lens_dict[f"lens_{i}"] = af.Model(
        al.Galaxy,
        redshift=0.5,
        bulge=bulge,
        mass=mass,
    )

# External Shear (an `al.MassField`, in the model's `fields` slot below):

field = af.Model(al.MassField, redshift=0.5, shear=af.Model(al.mp.ExternalShear))

# Extra Galaxies:

extra_galaxies_list = []

for centre in extra_galaxies_centres:

    # Extra Galaxy Light

    bulge = al.model_util.mge_model_from(
        mask_radius=mask_radius,
        total_gaussians=10,
        centre_fixed=centre,
        sigma_min=dataset.pixel_scales[0] / 10.0,
    )

    # Extra Galaxy Mass

    mass = af.Model(al.mp.dPIEMassSph)

    mass.centre = centre
    mass.sigma = af.UniformPrior(lower_limit=0.0, upper_limit=300.0)
    mass.r_core = 0.0  # vanishing core — fixed; the dPIE is analytic at r_core = 0
    mass.r_cut = 10.0  # truncation fixed at a fiducial radius
    mass.redshift_object = 0.5
    mass.redshift_source = 1.0
    mass.H0 = 67.66  # pinned: model constants, not parameters to sample
    mass.Om0 = 0.30966

    # Extra Galaxy

    extra_galaxy = af.Model(al.Galaxy, redshift=0.5, bulge=bulge, mass=mass)

    extra_galaxies_list.append(extra_galaxy)

extra_galaxies = af.Collection(extra_galaxies_list)

# Source:

bulge = al.model_util.mge_model_from(
    mask_radius=mask_radius,
    total_gaussians=20,
    gaussian_per_basis=1,
    centre_prior_is_uniform=False,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=bulge)

# Overall Lens Model:

model = af.Collection(
    galaxies=af.Collection(**lens_dict, source=source),
    fields=field,
    extra_galaxies=extra_galaxies,
)

__Over Sampling__

Over sampling is a numerical technique where the images of light profiles and galaxies are evaluated
on a higher resolution grid than the image data to ensure the calculation is accurate.

For lensing calculations, the high magnification regions of a lensed source galaxy require especially high levels of
over sampling to ensure the lensed images are evaluated accurately.

For a new user, the details of over-sampling are not important, therefore just be aware that calculations either:

 (i) use adaptive over sampling for the foregorund lens's light, which ensures high accuracy across.
 (ii) retain 2x2 over sampling in the outer regions of the image, where the lensed source's light appears,
 ensuring compact source profiles (e.g. a Multi Gaussian Expansion) are evaluated accurately.

Over sampling at each galaxy centre (both main lens galaxies and extra galaxies) is performed to ensure the lens
calculations are accurate across the full field of the group.

Once you are more experienced, you should read up on over-sampling in more detail via
the `autolens_workspace/*/guides/advanced/over_sampling.ipynb` notebook.

In [ ]:
all_centres = list(main_lens_centres) + list(extra_galaxies_centres)

over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=dataset.grid,
    sub_size_list=[4, 2, 2],
    radial_list=[0.3, 0.6],
    centre_list=all_centres,
)

dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

The imaging subplot updates the bottom two panels to reflect the update to over sampling, which now uses a higher
values in the centre.

Whilst you may not yet understand the details of over-sampling, you can at least track it visually in the plots
and later learnt more about it in the `over_sampling.ipynb` guide.

In [ ]:
aplt.subplot_imaging_dataset(dataset=dataset)

The `info` attribute shows the model in a readable format.

This shows the group scale model, with separate entries for each main lens galaxy, the source galaxy and the
extra galaxies.

In [ ]:
print(model.info)

The improved model has the same galaxies and profiles, but every `Sersic` is replaced by an MGE basis of
Gaussians whose `intensity` is solved by the inversion at every likelihood evaluation.

In [ ]:
af.ModelPlotter(model).figure()

__Search__

The lens model is fitted to the data using a non-linear search.

All examples in the autolens workspace use the nested sampling algorithm
Nautilus (https://nautilus-sampler.readthedocs.io/en/latest/), which extensive testing has revealed gives the most
accurate and efficient modeling results.

This example uses `Nautilus` because it returns the **full posterior** — every parameter's errors and the
covariances between them. The folder's `start_here.py` instead fits with `af.MultiStartProdigy`, a multi-start
gradient optimizer which is far faster but returns only a single best-fit model with no errors at all. Use that
one to check a model quickly, and this one when you need results you can quote.

Nautilus has one main setting that trades-off accuracy and computational run-time, the number of `live_points`.
A higher number of live points gives a more accurate result, but increases the run-time. A lower value give
less reliable lens modeling (e.g. the fit may infer a local maxima), but is faster.

The suitable value depends on the model complexity whereby models with more parameters require more live points.
The default value of 200 is sufficient for the vast majority of common lens models. Lower values often given reliable
results though, and speed up the run-times. In this example, given the model is quite simple (N=21 parameters), we
reduce the number of live points to 100 to speed up the run-time.

__Unique Identifier__

In the path above, the `unique_identifier` appears as a collection of characters, where this identifier is generated
based on the model, search and dataset that are used in the fit.

An identical combination of model and search generates the same identifier, meaning that rerunning the script will use
the existing results to resume the model-fit. In contrast, if you change the model or search, a new unique identifier
will be generated, ensuring that the model-fit results are output into a separate folder.

We additionally want the unique identifier to be specific to the dataset fitted, so that if we fit different datasets
with the same model and search results are output to a different folder. We achieve this below by passing
the `dataset_name` to the search's `unique_tag`.

__Iterations Per Update__

Every `iterations_per_quick_update`, the non-linear search outputs the maximum likelihood model and its best fit
image to the Jupyter Notebook display and to hard-disk.

This process takes around ~10 seconds, so we don't want it to happen too often so as to slow down the overall
fit, but we also want it to happen frequently enough that we can track the progress.

The value of 10000 below means this output happens every few minutes on GPU and every ~10 minutes on CPU, a good balance.

__Live Visual Update__

By default the quick-update image is only written to disk. Set `live_visual_update=True` to also push it to a
live display surface:

- **Python script** — a matplotlib window opens automatically and refreshes with each quick update, so you can
  watch the fit converge without leaving your terminal.
- **Jupyter / Colab notebook** — the cell that ran `search.fit(...)` shows a single self-updating image that
  refreshes in place every `iterations_per_quick_update`.

The disk write (`fit.png`) always happens regardless of this flag. Set it to `False` (the default) if you just
want the on-disk output, or if you are running in a headless environment (e.g. an HPC cluster).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("group"),  # The path where results and output are stored.
    name="modeling",  # The name of the fit and folder results are output to.
    unique_tag=dataset_name,  # A unique tag which also defines the folder.
    n_live=150,  # The number of Nautilus "live" points, increase for more complex models.
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    iterations_per_quick_update=10000,  # Every N iterations the max likelihood model is visualized in the Jupter Notebook and output to hard-disk.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Analysis__

We next create an `AnalysisImaging` object, which can be given many inputs customizing how the lens model is
fitted to the data (in this example they are omitted for simplicity).

Internally, this object defines the `log_likelihood_function` used by the non-linear search to fit the model to
the `Imaging` dataset.

It is not vital that you as a user understand the details of how the `log_likelihood_function` fits a lens model to
data, but interested readers can find a step-by-step guide of the likelihood
function at ``autolens_workspace/*/imaging/likelihood_function`

__JAX__

`AnalysisImaging` defaults to `use_jax=True`. For group-scale fits the
JAX speedup is substantial — the multi-galaxy deflection sum dominates
runtime on CPU but vectorises cleanly on GPU. Search driver wraps the
likelihood in `jax.vmap(jax.jit(...))`. Force NumPy with `use_jax=False`
(or `PYAUTO_DISABLE_JAX=1`) when debugging.

In [ ]:
analysis = al.AnalysisImaging(
    dataset=dataset,
    use_jax=True,  # JAX will use GPUs for acceleration if available, else JAX will use multithreaded CPUs.
)

__VRAM Use__

When running AutoLens with JAX on a GPU, the analysis must fit within the GPU's
available VRAM. If insufficient VRAM is available, the analysis will fail with an
out-of-memory error, typically during JIT compilation or the first likelihood call.

Two factors dictate the VRAM usage of an analysis:

- The number of arrays and other data structures JAX must store in VRAM to fit the model
  to the data in the likelihood function. This is dictated by the model complexity and dataset size.
  For a MGE model its relatively low, but for other models (e.g. pixelized sources) it can be much higher.

- The `batch_size` sets how many likelihood evaluations are performed simultaneously.
  Increasing the batch size increases VRAM usage but can reduce overall run time,
  while decreasing it lowers VRAM usage at the cost of slower execution.

Before running an analysis, users should check that the estimated VRAM usage for the
chosen batch size is comfortably below their GPU's total VRAM.

The method below prints the VRAM usage estimate for the analysis and model with the specified batch size,
it takes about 20-30 seconds to run so you may want to comment it out once you are familiar with your GPU's VRAM limits.

For a MGE model with the low resolution dataset fitted in this example VRAM use is relatively quite high (~1.8GB),
illustrating how group scale modeling is already quite VRAM intensive. For more complex models (e.g. pixelized sources)
and higher resolution datasets it can be much higher (> 1GB going beyond 10GB).

In [ ]:
analysis.print_vram_use(model=model, batch_size=search.batch_size)

__Run Times__

Lens modeling can be a computationally expensive process. When fitting complex models to high resolution datasets
run times can be of order hours, days, weeks or even months.

Run times are dictated by two factors:

 - The log likelihood evaluation time: the time it takes for a single `instance` of the lens model to be fitted to
   the dataset such that a log likelihood is returned.

 - The number of iterations (e.g. log likelihood evaluations) performed by the non-linear search: more complex lens
   models require more iterations to converge to a solution.

For this analysis, the log likelihood evaluation time is < 0.005 seconds on GPU, < 0.05 seconds on CPU, which is
extremely fast for group-scale lens modeling.

To estimate the expected overall run time of the model-fit we multiply the log likelihood evaluation time by an
estimate of the number of iterations the non-linear search will perform, which is around 20000 to 40000 for this model.

GPU run times are around 15 minutes, CPU run times are around 45 minutes.

__Model-Fit__

We begin the model-fit by passing the model and analysis object to the non-linear search (checkout the output folder
for on-the-fly visualization and results).

**Run Time Error:** On certain operating systems (e.g. Windows, Linux) and Python versions, the code below may produce
an error. If this occurs, see the `autolens_workspace/guides/modeling/bug_fix` example for a fix.

In [ ]:
print(
    """
    The non-linear search has begun running.

    This Jupyter notebook cell with progress once the search has completed - this could take a few minutes!

    On-the-fly updates every iterations_per_quick_update are printed to the notebook.
    """
)

result = search.fit(model=model, analysis=analysis)

print("The search has finished run - you may now continue the notebook.")

__Output Folder Layout__

Now the fit is running you should checkout the `autolens_workspace/output` folder. This is where results are
written to hard-disk in human-readable formats — `.json`, `.csv`, `.fits`, `.png` and plain text.

As the fit progresses, results are written on the fly using the highest likelihood model found by the
non-linear search so far. This means you can inspect the model-fit as it runs, without waiting for the
non-linear search to terminate.

Each completed fit lives at a path like::

    output/group/<dataset_name>/modeling/<unique_hash>/
        files/                         <- JSON + CSV: loadable Python objects
            tracer.json                <- max log likelihood Tracer (all lens galaxies + source)
            model.json                 <- fitted af.Collection model
            samples.csv                <- full Nautilus samples
            samples_summary.json       <- max log likelihood parameter values + errors
            samples_info.json          <- metadata about the samples
            search.json                <- non-linear search configuration
            settings.json              <- search settings
            cosmology.json             <- cosmology used for the fit
            covariance.csv             <- parameter covariance matrix
        image/                         <- FITS + PNG: imaging products
            dataset.fits               <- data, noise-map and PSF
            fit.fits                   <- model image, residuals, chi-squared map
            tracer.fits                <- tracer image-plane images per galaxy
            source_plane_images.fits   <- source plane reconstructions
            model_galaxy_images.fits   <- per-galaxy model images (lens_0, lens_1, ..., source)
            galaxy_images.fits         <- per-galaxy images
            dataset.png, fit.png, tracer.png   <- visualisations
        model.info                     <- human-readable model summary
        model.results                  <- human-readable fit summary
        search.summary                 <- search run summary
        search_internal/               <- internal files used to resume / visualise the search
        metadata                       <- run metadata

The `<unique_hash>` is a 32-character identifier derived from the model, search and dataset, so re-running the
same configuration resumes from the existing fit automatically.

__Result__

The search returns a result object, which whose `info` attribute shows the result in a readable format.

The result contains entries for each main lens galaxy (e.g. `lens_0`), the source galaxy and the extra galaxies.

[Above, we discussed that the `info_whitespace_length` parameter in the config files could b changed to make
the `model.info` attribute display optimally on your computer. This attribute also controls the whitespace of the
`result.info` attribute.]

In [ ]:
print(result.info)

The `Result` object also contains:

 - The model corresponding to the maximum log likelihood solution in parameter space.
 - The corresponding maximum log likelihood `Tracer` and `FitImaging` objects.

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.

In [ ]:
print(result.max_log_likelihood_instance)

aplt.subplot_tracer(tracer=result.max_log_likelihood_tracer, grid=result.grids.lp)

aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

It also contains information on the posterior as estimated by the non-linear search (in this example `Nautilus`).

Below, we make a corner plot of the "Probability Density Function" of every parameter in the model-fit.

The plot is labeled with short hand parameter names (e.g. `sersic_index` is mapped to the short hand
parameter `n`). These mappings ate specified in the `config/notation.yaml` file and can be customized by users.

The superscripts of labels correspond to the name each component was given in the model (e.g. for the `Isothermal`
mass its name `mass` defined when making the `Model` above is used).

In [ ]:
aplt.corner_anesthetic(samples=result.samples)

This script gives a concise overview of the basic modeling API, fitting one the simplest lens models possible.

Lets now consider what features you should read about to improve your lens modeling, especially if you are aiming
to fit more complex models to your data.

This is especially important for group scale modeling, in order to reduce the complexity of the model.

__Features__

The examples in the `autolens_workspace/*/imaging/features` package illustrate other lens modeling features.

We recommend you checkout the following features, because they make lens modeling in general more reliable and
efficient (you will therefore benefit from using these features irrespective of the quality of your data and
scientific topic of study).

We recommend you now checkout the following features:

- ``scaling_relation``: This feature allows you to model the light and mass of the extra galaxies using a scaling relation.
- ``group_halo``: Fit the same data with and without a group-scale dark-matter halo (with truncated dPIE members),
  teaching whether your group needs one.
- ``linear_light_profiles``: The model light profiles use linear algebra to solve for their intensity, reducing model complexity.
- ``multi_gaussian_expansion``: The lens (or source) light is modeled as ~25-100 Gaussian basis functions
- ``pixelization``: The source is reconstructed using an adaptive rectangular or Delaunay mesh
- ``no_lens_light``: The foreground lens's light is not present in the data and thus omitted from the model.

For group scale modeling, the multi Gaussian expansion is particularly important, as this can dramatically reduce the
dimensionality of the model and improve the accuracy of the fit for both the lens and source galaxies.

It is also recommended you read through the `imaging` package, to get a complete picture of how point-source
modeling works.

__Data Preparation__

If you are looking to fit your own CCD imaging data of a strong lens, checkout
the `autolens_workspace/*/imaging/data_preparation/start_here.ipynb` script for an overview of how data should be
prepared before being modeled.

__HowToLens__

This `start_here.py` script, and the features examples above, do not explain many details of how lens modeling is
performed, for example:

 - How does PyAutoLens perform ray-tracing and lensing calculations in order to fit a lens model?
 - How is a lens model fitted to data? What quantifies the goodness of fit (e.g. how is a log likelihood computed?).
 - How does Nautilus find the highest likelihood lens models? What exactly is a "non-linear search"?

You do not need to be able to answer these questions in order to fit lens models with PyAutoLens and do science.
However, having a deeper understanding of how it all works is both interesting and will benefit you as a scientist

This deeper insight is offered by the **HowToLens** Jupyter notebook lectures, which live in their own repository:
https://github.com/PyAutoLabs/HowToLens.

I recommend that you check them out if you are interested in more details!

__Modeling Customization__

The folders `autolens_workspace/*/guides/modeling/searches` gives an overview of alternative non-linear searches,
other than Nautilus, that can be used to fit lens models.

They also provide details on how to customize the model-fit, for example the priors.